In [103]:
import pandas as pd
import numpy as np

In [104]:
def load_data(url):
    
    df = pd.read_csv(url)
    
    return df

In [105]:
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"

df = load_data(url)

print(df.shape)
print(df.head())
print(df.dtypes)

(10000, 11)
   model_year  origin fuel_type         drivetrain  num_doors  \
0        2006  Europe  Gasoline  Front-wheel drive          4   
1        2008  Europe    Diesel  Front-wheel drive          4   
2        1996    Asia  Gasoline  Front-wheel drive          5   
3        1989  Europe  Gasoline  Front-wheel drive          4   
4        1994     USA    Diesel  Front-wheel drive          3   

   engine_displacement  num_cylinders  horsepower  vehicle_weight  \
0                 2180              6       243.0            3870   
1                 2390              6       272.0            4210   
2                 2320              6       267.0            4240   
3                 2130              6       258.0            4490   
4                 2580              7       304.0            4510   

   acceleration  fuel_efficiency_mpg  
0           NaN                 31.9  
1           NaN                 31.3  
2          17.3                 27.5  
3          18.7           

In [111]:
def split_data(df, seed=42):
    
    n = len(df)
    
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test
    
    np.random.seed(seed)
    
    idx = np.arange(n)
    np.random.shuffle(idx)
    
    df_train = df.iloc[
        idx[:n_train]
    ]
    
    df_val = df.iloc[
        idx[n_train:n_train + n_val]
    ]
    
    df_test = df.iloc[
        idx[n_train + n_val:]
    ]
    
    return df_train, df_val, df_test

In [112]:
def prepare_X(df, target):
    
    X = df.drop(
        columns=[target]
    ).copy()
    
    return X

In [113]:
def prepare_y(df, target):
    
    y = df[target].values
    
    return y

In [147]:
def prepare_features(
    X_train,
    X_val,
    X_test
):
    
    X_train = X_train.copy()
    X_val = X_val.copy()
    X_test = X_test.copy()
    
    
    # --------------------------------------------------------
    # Identify numerical and categorical columns
    # using TRAINING DATA ONLY
    # --------------------------------------------------------
    
    numerical = X_train.select_dtypes(
        include=np.number
    ).columns.tolist()
    
    categorical = X_train.select_dtypes(
        exclude=np.number
    ).columns.tolist()
    
    
    # --------------------------------------------------------
    # Handle numerical missing values
    # using TRAINING MEAN ONLY
    # --------------------------------------------------------
    
    for column in numerical:
        
        mean_value = X_train[column].mean()
        
        X_train[column] = X_train[column].fillna(
            mean_value
        )
        
        X_val[column] = X_val[column].fillna(
            mean_value
        )
        
        X_test[column] = X_test[column].fillna(
            mean_value
        )
    
    
    # --------------------------------------------------------
    # Handle categorical missing values
    # BEFORE one-hot encoding
    # --------------------------------------------------------
    
    for column in categorical:
        
        X_train[column] = X_train[column].fillna(
            'missing'
        )
        
        X_val[column] = X_val[column].fillna(
            'missing'
        )
        
        X_test[column] = X_test[column].fillna(
            'missing'
        )
    
    
    # --------------------------------------------------------
    # One-hot encoding (manual)
    #
    # Training data determines the categories.
    # --------------------------------------------------------
    
    for column in categorical:
        
        # Categories from TRAINING set only
        values = list(X_train[column].value_counts().index)
        
        for value in values:
            
            col_name = '%s_%s' % (column, value)
            
            X_train[col_name] = (X_train[column] == value).astype(float)
            X_val[col_name]   = (X_val[column] == value).astype(float)
            X_test[col_name]  = (X_test[column] == value).astype(float)
        
        # Drop the original categorical column
        X_train = X_train.drop(columns=[column])
        X_val   = X_val.drop(columns=[column])
        X_test  = X_test.drop(columns=[column])
    
    
    # --------------------------------------------------------
    # Make validation and test have EXACTLY the same
    # columns as training
    # --------------------------------------------------------
    
    X_val = X_val.reindex(
        columns=X_train.columns,
        fill_value=0
    )
    
    X_test = X_test.reindex(
        columns=X_train.columns,
        fill_value=0
    )
    
    
    # --------------------------------------------------------
    # Convert to numpy
    # --------------------------------------------------------
    
    X_train = X_train.values.astype(float)
    X_val   = X_val.values.astype(float)
    X_test  = X_test.values.astype(float)
    
    
    # --------------------------------------------------------
    # Add intercept
    # --------------------------------------------------------
    
    X_train = np.column_stack(
        [
            np.ones(len(X_train)),
            X_train
        ]
    )
    
    X_val = np.column_stack(
        [
            np.ones(len(X_val)),
            X_val
        ]
    )
    
    X_test = np.column_stack(
        [
            np.ones(len(X_test)),
            X_test
        ]
    )
    
    return X_train, X_val, X_test

In [148]:
def linear_regression(X, y, r=0):
    
    # X transpose
    XT = X.T
    
    # X^T X
    XTX = XT @ X
    
    # X^T y
    XTy = XT @ y
    
    # Regularization matrix
    reg = np.eye(
        X.shape[1]
    ) * r
    
    # Do NOT regularize the intercept
    reg[0, 0] = 0
    
    # Explicit normal equation
    w = np.linalg.inv(
        XTX + reg
    ) @ XTy
    
    return w

In [149]:
def predict(X, w):
    
    return X @ w

In [150]:
def rmse(y, y_pred):
    
    error = y_pred - y
    
    mse = np.mean(
        error ** 2
    )
    
    return np.sqrt(mse)

In [151]:
def train_and_evaluate(
    df_train,
    df_val,
    target,
    r=0
):
    
    # Prepare X
    X_train_raw = prepare_X(
        df_train,
        target
    )
    
    X_val_raw = prepare_X(
        df_val,
        target
    )
    
    # We don't need test here
    X_test_raw = X_val_raw.copy()
    
    
    # Prepare feature matrices
    X_train, X_val, _ = prepare_features(
        X_train_raw,
        X_val_raw,
        X_test_raw
    )
    
    
    # Prepare target
    y_train = prepare_y(
        df_train,
        target
    )
    
    y_val = prepare_y(
        df_val,
        target
    )
    
    
    # Train
    w = linear_regression(
        X_train,
        y_train,
        r
    )
    
    
    # Predict
    y_pred = predict(
        X_val,
        w
    )
    
    
    # RMSE
    score = rmse(
        y_val,
        y_pred
    )
    
    return score

In [152]:
def test_regularization(
    df_train,
    df_val,
    target,
    r_values
):
    
    results = []
    
    for r in r_values:
        
        score = train_and_evaluate(
            df_train,
            df_val,
            target,
            r
        )
        
        results.append(
            (r, score)
        )
    
    print("Regularization results")
    print("-" * 40)
    
    for r, score in results:
        
        print(
            f"r = {r:<8} "
            f"RMSE = {score:.4f}"
        )
    
    
    # Best RMSE
    best_r, best_score = min(
        results,
        key=lambda x: x[1]
    )
    
    print()
    print("Best r:", best_r)
    print(
        "Best RMSE:",
        round(best_score, 4)
    )
    r = best_r
    return results 
    

In [153]:
def test_seeds(
    df,
    target,
    seeds,
    r=1
):
    
    scores = []
    
    for seed in seeds:
        
        df_train, df_val, df_test = split_data(
            df,
            seed
        )
        
        score = train_and_evaluate(
            df_train,
            df_val,
            target,
            r
        )
        
        scores.append(score)
        
        print(
            f"Seed {seed}: "
            f"RMSE = {score:.6f}"
        )
    
    
    mean_score = np.mean(scores)
    std_score = np.std(scores)
    
    
    print()
    print(
        "Mean RMSE:",
        round(mean_score, 6)
    )
    
    print(
        "Standard deviation:",
        round(std_score, 6)
    )
    
    return scores, std_score

In [154]:
df_train, df_val, df_test = split_data(
    df,
    seed=42
)

In [155]:
r_values = [
    0,
    0.00001,
    0.0001,
    0.001,
    0.01,
    0.1,
    1,
    10,
    100
]

results = test_regularization(
    df_train,
    df_val,
    'fuel_efficiency_mpg',
    r_values
)

Regularization results
----------------------------------------
r = 0        RMSE = 2471.2137
r = 1e-05    RMSE = 1.6288
r = 0.0001   RMSE = 1.6288
r = 0.001    RMSE = 1.6288
r = 0.01     RMSE = 1.6288
r = 0.1      RMSE = 1.6288
r = 1        RMSE = 1.6288
r = 10       RMSE = 1.6288
r = 100      RMSE = 1.6323

Best r: 1
Best RMSE: 1.6288


In [156]:
r 

1

In [157]:
seeds = [
    0, 1, 2, 3, 4,
    15, 26, 37, 48, 59
]

scores, std = test_seeds(
    df,
    'fuel_efficiency_mpg',
    seeds,
    r
)

Seed 0: RMSE = 1.626596
Seed 1: RMSE = 1.624519
Seed 2: RMSE = 1.634802
Seed 3: RMSE = 1.640308
Seed 4: RMSE = 1.586660
Seed 15: RMSE = 1.586001
Seed 26: RMSE = 1.598601
Seed 37: RMSE = 1.574283
Seed 48: RMSE = 1.622878
Seed 59: RMSE = 1.591333

Mean RMSE: 1.608598
Standard deviation: 0.022452


In [158]:
X_train_raw = prepare_X(
    df_train,
    'fuel_efficiency_mpg'
)

X_val_raw = prepare_X(
    df_val,
    'fuel_efficiency_mpg'
)

X_test_raw = prepare_X(
    df_test,
    'fuel_efficiency_mpg'
)

X_train, X_val, X_test = prepare_features(
    X_train_raw,
    X_val_raw,
    X_test_raw
)

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)

X_train shape: (6000, 17)
X_val shape: (2000, 17)
X_test shape: (2000, 17)
